# Arctic sea-ice thickness — a polar L4 map

Demonstrates the **surface-only polar-map pattern**: pull one day of
Arctic sea-ice thickness from the merged CryoSat-2 / SMOS L4 product and
render it as a map over the Arctic basin.

Dataset `esa_obs-si_arc_phy-sit_nrt_l4-multi_P1D-m` is near-real-time
and surface-only (no depth axis). Sea-ice **thickness** retrieval from
altimetry is winter-weighted — coverage is densest Oct–Apr — and the
NRT window rolls forward, so we probe a recent date and note that a
deep-summer re-run may need an earlier (winter) date.

Reads credentials from `COPERNICUSMARINE_SERVICE_USERNAME` /
`COPERNICUSMARINE_SERVICE_PASSWORD`.

## Setup

The imports: `numpy` and `matplotlib` for the array math and the map,
`pyramids` plus pyramids' `NetCDF` for reading the downloaded L4 file, and
`earthlens` for the unified `EarthLens` entry point and the CMEMS
`Catalog`. The output directory is created up front.

In [ ]:
import datetime as dt
import os
from pathlib import Path

import numpy as np
from pyramids.dataset import Dataset, GeoReference
from pyramids.netcdf import NetCDF
from pyramids.plot import ColorBar

from earthlens.cmems import Catalog
from earthlens.core import EarthLens

OUT_DIR = Path('data/cmems-seaice')
OUT_DIR.mkdir(parents=True, exist_ok=True)

### Request parameters

The dataset id is fixed; the probe date is computed ~45 days back to stay
inside the rolling NRT window. In deep summer (when thickness coverage
thins out) bump this to a winter date such as `'2024-02-15'`.

In [ ]:
DATASET_ID = 'esa_obs-si_arc_phy-sit_nrt_l4-multi_P1D-m'
# ~45 days back keeps us inside the rolling NRT window; in deep summer
# bump this to a winter date (e.g. '2024-02-15') where SIT coverage is dense.
PROBE_DATE = (dt.datetime.now(dt.UTC) - dt.timedelta(days=45)).strftime('%Y-%m-%d')

### Inspect the catalog entry

Before downloading, look up the dataset in the CMEMS `Catalog` to confirm
its domain and cadence and to see which variables it exposes.

In [ ]:
ds_meta = Catalog().get_dataset(DATASET_ID)
print(ds_meta)
print(f'domain: {ds_meta.domain}')
print('variables:', sorted(ds_meta.variables))
print('probe date:', PROBE_DATE)

## Download one day over the Arctic basin

Build the `EarthLens` request first — source, date window, cadence,
dataset, the single `sea_ice_thickness` variable, an Arctic-basin bounding
box, the output path, and the CMEMS credentials from the environment.

In [ ]:
el = EarthLens(
    data_source='cmems',
    start=PROBE_DATE,
    end=PROBE_DATE,
    cadence='daily',
    dataset=DATASET_ID,
    variables=['sea_ice_thickness'],
    aoi=[-180.0, 65.0, 180.0, 88.0],
    path=OUT_DIR,
    service_username=os.environ.get('COPERNICUSMARINE_SERVICE_USERNAME'),
    service_password=os.environ.get('COPERNICUSMARINE_SERVICE_PASSWORD'),
)

With the request built, `download()` fetches the subset and returns the
list of written NetCDF paths.

In [ ]:
paths = el.download()
print(paths)

## Open and inspect the field

The L4 grid is a polar stereographic projection, so the array carries
`xc` / `yc` projected axes (plus 2-D lat/lon). We read the file through
pyramids' `NetCDF`, decode it into an `pyramids` dataset, and close the
handle.

In [ ]:
nc = NetCDF.read_file(paths[0], read_only=True)
print('variables :', nc.variable_names)
print('dimensions:', nc.dimension_sizes)

Collapse the single time step to get the 2-D thickness field, then
summarise the valid (non-NaN) cells.

In [ ]:
sit_var = nc.get_variable('sea_ice_thickness')
scale = float(np.asarray(sit_var.scale)[0])
offset = float(np.asarray(sit_var.offset)[0])
# apply the CF packing before reporting metres (serapeum-org/pyramids#1124)
sit = sit_var.read_array(masked=True).astype('float64')[0] * scale + offset
print(f'valid cells: {sit.count()}')
if sit.count():
    print(
        f'thickness min/mean/max: {float(sit.min()):.2f} / '
        f'{float(sit.mean()):.2f} / {float(sit.max()):.2f} m'
    )

## Map the thickness field

A plain pyramids of the projected grid — thicker multi-year ice
north of the Canadian Arctic Archipelago, thinner first-year ice toward
the marginal seas.

In [ ]:
field = Dataset.from_array(
    sit.filled(np.nan),
    no_data_value=np.nan,
    geo_ref=GeoReference(geo=sit_var.geotransform, epsg=sit_var.epsg),
)
field.plot(
    cmap='Blues',
    vmin=0,
    vmax=5,
    colorbar=ColorBar(label='Sea-ice thickness (m)'),
    title=f'Arctic sea-ice thickness (L4 NRT, {PROBE_DATE})',
)
nc.close()